# How to select good initial mapping

1. Sabre selects an initial mapping $g$ for a circuit $C$ by starting from a random mapping $f$, sending $f$ through $C+C^{-1}+\ldots + C+C^{-1}$ and using the final mapping $g$ as the official intial mapping. Tbiis is effective perhaps due to in this way it can get the global structure of the circuit in an economical way. 

2. FiDLS uses two initial mapping methods based on subgraph isomorphism: TopGraph and WgtGraph. ToPGraph uses any embedding of the first section of $C$ as the initial mapping, where the first section of $C$ is a maximal prelist of $C$ whose interaction graph is embeddable into $AG$. WgtGraph considers the weighted version of interaction graph. It tries to find a subgraph of $AG$ which has the maximal weight. 

TopGraph has the following shortcomings:
a. There are many different embeddings of a subgraph. We need select the 'best' one.
b. There is no reason that we need to start from the first section. Perhaps a middle section has the largest size.

WgtGraph takes the global information into consideration, but it has the following limitations:
a. It treats every occurrence of a CX as the same. We should attach a different weight to each occurrence according to its layer number.
b. There is no reason that we need to start from the very beginning of the circuit.

# TODO
1. For circuits with small width, run Sabre 100 times and record the best initial mapping. Examine what property these initial mappings have.
2. Find the best place and mapping to insert using WgtGraph.

For each edge $(p,q)$, we associated with a list $L_{p,q}$ of layers in which either CX(p,q) or CX(q,p) appears. We define a weight function $W(s,p,q) = \sum_{\ell \in L_{p,q}} d-(\ell\ominus s) $, where $\ell\ominus s = d-|\ell-s|$ and $d$ is the depth of $C$. For each $s$, we find the subgraph of $AG$ which has the maximum weight $w_s$; and then select the $s$ with the maximum $w_s$.

In [ ]:
def best_wtg_o_ini_mapping(C, G, anchor, stop): #'o' for original
    ''' Return a graph g which is isomorphic to a subgraph of G
            while maximizing the number of CNOTs in C that correspond to edges in g
        Method: sort the edges according to their weights (the number of CNOTs in C corresponding to each edge);
                construct a graph by starting with the edge with the largest weight; then consider the edge with the second large weight, ...
                if in any step the graph is not isomorphic to a subgraph of G, skip this edge and consider the next till all edges are considered.
    
    Args:
        C (list): the input circuit
        G (graph): the architecture graph
        
    Returns:
        g (graph)
        map (dict)
    '''    
    g_of_c = graph_of_circuit(C)
    test = is_embeddable(g_of_c, G, anchor, stop)
    if test[0]:
        #print('The graph of the circuit is embeddable in G')
        return g_of_c, test[1]
    
    edge_wgt_list = list([C.count([e[0],e[1]]) + C.count([e[1],e[0]]), e] for e in g_of_c.edges())
    edge_wgt_list.sort(key=lambda t: t[0], reverse=True) # q[0] weight, q[1] edge
    
    '''Sort the edges reversely according to their weights''' 
    EdgeList = list(item[1] for item in edge_wgt_list)    
    #edge_num = len(EdgeList)
    
    '''We search backward, remove the first edge that makes g not embeddable, 
            and continue till all edges are evaluated in sequence. '''
            
    #Hard_Edge_index = 0 # the index of the first hard edge
    g = nx.Graph()
    result = dict()
    # add the first edge into g
    edge = EdgeList[0]
    g.add_edge(edge[0], edge[1])
    
    # rp = 0
    # for rp in range(edge_num):
    #     # h is the index of the last edge that can be added into g
    #     h = Hard_Edge_index
    #     if h == edge_num - 1: 
    #         return g, result
        
    #     EdgeList_temp = EdgeList[h+1:edge_num]
    #     for edge in EdgeList_temp:           
    #         g.add_edge(edge[0], edge[1])           
    #         i = EdgeList.index(edge)            
    #         # find the largest i such that the first i-1 edges are embeddable
    #         test = is_embeddable(g, G, anchor, stop)
    #         if not test[0]:
    #             Hard_Edge_index = i
    #             g.remove_edge(edge[0], edge[1])
    #             break
    #         result = test[1]
    #         if i == edge_num- 1:                
    #             return g, result
    # return g, result
    

    #EdgeList_temp = EdgeList[:]
    for edge in EdgeList:           
        g.add_edge(edge[0], edge[1])           
        test = is_embeddable(g, G, anchor, stop)
        if not test[0]:
            g.remove_edge(edge[0], edge[1])
            if nx.degree(g, edge[0]) == 0: g.remove_node(edge[0])
            if nx.degree(g, edge[1]) == 0: g.remove_node(edge[1])
        else:
            result = test[1]
    return g, result

In [4]:
from collections import defaultdict
import copy
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.dagcircuit import DAGCircuit, DAGOpNode, DAGInNode, DAGOutNode
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.transpiler.layout import Layout
from qiskit.circuit.quantumregister import Qubit
from qiskit.visualization import dag_drawer, circuit_drawer
from qiskit.qasm import Qasm

import matplotlib.pyplot as plt
import networkx as nx
import rustworkx as rx

from dac_part import is_rx_embeddable, is_embeddable, draw_nx_graph

In [5]:
# Weighted SUBGRAPH initial mapping
#TODO: We should also calculate where to insert the mapping

def wtggraph(dag: DAGCircuit, AG: nx.Graph): 
    ''' Return a graph g which is isomorphic to a subgraph of AG
            while maximizing the number of CNOTs in C that correspond to edges in g
        Method: sort the edges according to their weights (the number of CNOTs in C corresponding to each edge);
                construct a graph by starting with the edge with the largest weight; then consider the edge with the second large weight, ...
                if in any step the graph is not isomorphic to a subgraph of G, skip this edge and consider the next till all edges are considered.
    '''    
    g_of_c = layer_distribution_graph_of_dag(dag)
    test = is_embeddable(g_of_c, AG, 10)
    print(f'The graph of the circuit is embeddable in G? {test[0]}')
    if test[0]:
        print('The graph of the circuit is embeddable in G')
        return g_of_c, test[1]
    
    for edge in g_of_c.edges():
        print(edge, nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])    
    
    edge_wgt_list = list([len(nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge]), edge] for edge in g_of_c.edges())
    edge_wgt_list.sort(key=lambda t: t[0], reverse=True) # q[0] weight, q[1] edge'
    #print(edge_wgt_list)

    
    '''Sort the edges reversely according to their weights''' 
    EdgeList = list(item[1] for item in edge_wgt_list)    
    #edge_num = len(EdgeList)
    
    '''We search backward, remove the first edge that makes g not embeddable, 
            and continue till all edges are evaluated in sequence. '''
            
    g = nx.Graph()
    result = dict()
    
    # add the first edge into g
    edge = EdgeList[0]
    g.add_edge(edge[0], edge[1])
    num_gate_sat = len(nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])
    
    for edge in EdgeList:           
        g.add_edge(edge[0], edge[1])           
        test = is_embeddable(g, AG, 10)
        if not test[0]:
            g.remove_edge(edge[0], edge[1])
            if nx.degree(g, edge[0]) == 0: g.remove_node(edge[0])
            if nx.degree(g, edge[1]) == 0: g.remove_node(edge[1])
        else:
            result = test[1]
            num_gate_sat += len(nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])
    
    #! either (p,q) or (q,p) in nx.get_edge_attributes(g_of_c, 'layer_distribution')
    for edge in g.edges():
        if edge not in nx.get_edge_attributes(g_of_c, 'layer_distribution'):
            edge = (edge[1],edge[0])
        print(edge, nx.get_edge_attributes(g_of_c, 'layer_distribution')[edge])
    return g, result, num_gate_sat

In [8]:
import matplotlib.pyplot as plt
import networkx as nx
from dac_part import remove_1q_and_consecutive_2q_gates_in_circuit

from ag import qgrid, q20
import time
import os

path = '../bench/qiskit_circuit_benchmark/' 
#filename = 'excitation_preserving_6.qasm'
#filename = 'grover_operator_14.qasm'
#filename = 'quantum_volume_16.qasm'
#filename = 'phase_oracle_14.qasm'
#filename = 'qft_6.qasm'
filename = 'phase_estimation_6.qasm'

print(filename)
#AG = q20()
AG = qgrid(2,3)
rxAG = rx.networkx_converter(AG)


with open(path+filename, 'r') as file:
    qasm_code = file.read()

# Create a QuantumCircuit from the QASM code
qc = QuantumCircuit.from_qasm_str(qasm_code)
print(qc.count_ops())

newcirc = remove_1q_and_consecutive_2q_gates_in_circuit(qc)

dag = circuit_to_dag(newcirc)
print(filename, dag.count_ops(), dag.depth())

#g, _, Edge_Layer = graph_profile(newcirc, True)
#max_layer_num = dag.depth()
#selected_edges = []
#cutting_list = all_cutting_points(selected_edges, Edge_Layer, max_layer_num, g)
#print(f'The cutting points are {cutting_list}')
#selected_edges = [(3,4)]
#cutting_list = all_cutting_points(selected_edges, Edge_Layer, max_layer_num, g)
#print(f'The cutting points for {selected_edges} are {cutting_list}')


#dynamic_graph_partition(qc, lev=1)

phase_estimation_6.qasm
OrderedDict([('cx', 102), ('t', 60), ('tdg', 45), ('h', 30), ('u', 22), ('u3', 4)])
phase_estimation_6.qasm {'cx': 81} 80


In [9]:
gx, result, num_sat = wtggraph(dag, AG)
print(gx.nodes(), result, num_sat)

NameError: name 'layer_distribution_graph_of_dag' is not defined